In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error

In [22]:
combined = pd.read_csv("./finaldata/combined_events.csv")


In [23]:

missing_summary = combined.isnull().sum()

print("\nMissing Value Summary:")
print(missing_summary[missing_summary > 0].sort_values(ascending=False))

# Optionally, inspect how many rows have *any* missing entries
nan_rows = combined[combined.isnull().any(axis=1)]
print(f"\nTotal rows with at least one NaN: {len(nan_rows)}")

# (Optional) Drop rows or impute missing values if needed
# Example: fill AWARDED NaNs with median
combined['AWARDED'].fillna(combined['AWARDED'].median(), inplace=True)



Missing Value Summary:
Series([], dtype: int64)

Total rows with at least one NaN: 0


/tmp/ipykernel_6633/2177110586.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  combined['AWARDED'].fillna(combined['AWARDED'].median(), inplace=True)


In [ ]:


# ----------------------------------------------------------
# 2️⃣ Define feature matrix and target
# ----------------------------------------------------------
X = combined[['log_AWARDED', 'Event_Type', 'Location_Type',
              'A.S. Advertisement Pass', 'Day_of_Week']]
y = combined['log_Attendance_Ratio']

categorical_cols = ['Event_Type', 'Location_Type', 'A.S. Advertisement Pass', 'Day_of_Week']
numeric_cols = ['log_AWARDED']

# Preprocessing: one-hot encode categoricals, scale numeric
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ----------------------------------------------------------
alphas = np.logspace(-3, 3, 50)

ridge = Pipeline([
    ('prep', preprocessor),
    ('model', RidgeCV(alphas=alphas, cv=5))
])

lasso = Pipeline([
    ('prep', preprocessor),
    ('model', LassoCV(alphas=alphas, cv=5, max_iter=5000))
])

elastic = Pipeline([
    ('prep', preprocessor),
    ('model', ElasticNetCV(alphas=alphas, l1_ratio=[0.1, 0.5, 0.9], cv=5, max_iter=5000))
])

# Fit models
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)
elastic.fit(X_train, y_train)

# ----------------------------------------------------------
# 4️⃣ Evaluate performance
# ----------------------------------------------------------
models = {
    'Ridge': ridge,
    'Lasso': lasso,
    'ElasticNet': elastic
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    results.append({
        'Model': name,
        'R²': r2_score(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    })

results_df = pd.DataFrame(results)
print(results_df)

# ----------------------------------------------------------
# 5️⃣ Visualize model performance
# ----------------------------------------------------------
sns.barplot(x='Model', y='R²', data=results_df, palette='crest')
plt.title("Model Comparison: Ridge vs Lasso vs ElasticNet (R² Score)")
plt.show()

sns.barplot(x='Model', y='RMSE', data=results_df, palette='flare')
plt.title("Model Comparison: Ridge vs Lasso vs ElasticNet (RMSE)")
plt.show()

# ----------------------------------------------------------
# 6️⃣ Interpret coefficients (optional)
# ----------------------------------------------------------
# Extract feature names
encoder = ridge.named_steps['prep'].named_transformers_['cat']
encoded_cols = encoder.get_feature_names_out(categorical_cols)
all_features = np.concatenate([encoded_cols, numeric_cols])

coefs = pd.Series(
    ridge.named_steps['model'].coef_,
    index=all_features
).sort_values(key=abs, ascending=False)

print("\nTop Ridge Coefficients:")
print(coefs.head(10))
